In [ ]:
import numpy as np
import flammkuchen as fl
import napari
from pathlib import Path

from split_dataset import SplitDataset
import json

import tifffile as tiff

In [ ]:
master_path = Path(r"\\portulab.synology.me\data\Hagar and Ot\E0040\v10\LS")
fish_list = list(master_path.glob("*_f*"))

unexported = [f for f in fish_list if not (f / "regions_masks_mapzbrain.h5").exists()]
unexported

In [ ]:
########## load masks
path_to_masks = Path(r"Z:\Hagar and Ot\good h2b reference from mapzebrain")
ipn_mask_mapzbrain = tiff.imread(path_to_masks / 'interpeduncular_nucleus.tif')
ahb_mask_mapzbrain = tiff.imread(path_to_masks / 'superior_dorsal_medulla_oblongata_stripe_1_entire.tif')
pretectum_mask_mapzbrain = tiff.imread(path_to_masks / 'superior_dorsal_medulla_oblongata_stripe_1_entire.tif')

#abenula masks are manual due to weird abenula in mapzebrain

regions = [ipn_mask_mapzbrain, ahb_mask_mapzbrain, pretectum_mask_mapzbrain]

In [ ]:
d = {
    'l_habenula_coords': coords_to_keep[0],
    'r_habenula_coords': coords_to_keep[1],
    'ipn_coords': coords_to_keep[2],
    'ahb_coords': coords_to_keep[3],
    'pretectum_coords': coords_to_keep[4],
    'masks': masks,
}
fl.save(path / 'regions_masks.h5', d)

In [ ]:
for path in unexported:
    print(path)
    suite2p_data = fl.load(path / "data_from_suite2p_cells.h5")
    in_brain = fl.load(path / "data_from_suite2p_cells_brain.h5")['coords_idx']

    trans_coords = fl.load(path / 'registration' / 'to_h2b_baier_ref' / 'antspy' / 'mov_coords_transformed.h5')[in_brain]

    coords_to_keep = [[], [], [], [], []]
    num_cells = np.shape(trans_coords)[0]

    for region in range(len(regions) + 2): 
        print(region)
        tmp_coords_to_keep = []
        if region < len(regions):
            labels = regions[region]
            for i in range(num_cells):
                tmp_cell = labels[int(trans_coords[i, 2]), int(trans_coords[i, 1]), int(trans_coords[i, 0])]
                if tmp_cell > 0:
                    tmp_coords_to_keep = tmp_coords_to_keep + [i]

            coords_to_keep[region] = tmp_coords_to_keep
        
        else:
            print(region)
            manual_masks = fl.load(path / 'regions_masks.h5')
            if region is (len(regions) + 1):
                coords_to_keep[region] = manual_masks['l_habenula_coords']
            else:
                coords_to_keep[region] = manual_masks['r_habenula_coords']
        
    
    ########### saving the coords in the region
    d = {
        'ipn_coords': coords_to_keep[0],
        'ahb_coords': coords_to_keep[1],
        'pretectum_coords': coords_to_keep[1],
        'lhab_coords': coords_to_keep[1],
        'rhab_coords': coords_to_keep[1],
    }
    fl.save(path / 'regions_masks_mapzbrain.h5', d)


In [ ]:
print(len(coords_to_keep[1]))